# Lab 08-05 · Verilog Semantics and Testbench Coverage

Companion to chapter [08-05](../../chapters/08-05-verilog-and-testbenches.md). Handout: [README.md](README.md).

Small Python models of the Verilog rules that cause the most bugs (four-valued logic, blocking versus non-blocking assignment), and an experiment on random-test coverage. If Icarus Verilog is installed, the last part runs the lab's testbenches. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import itertools
import math
import shutil
import subprocess
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(805)
HAVE_IVERILOG = shutil.which("iverilog") is not None
print("iverilog available:", HAVE_IVERILOG)

iverilog available: True


## Part 1 · Four-valued logic

**🔮 Predict:** of the 16 pairs of values from {0, 1, X, Z}, how many give a definite AND? A definite XOR?

In [2]:
V = "01xz"
n = lambda a: "x" if a == "z" else a


def AND(a, b):
    a, b = n(a), n(b)
    return "0" if "0" in (a, b) else "1" if a == b == "1" else "x"


def OR(a, b):
    a, b = n(a), n(b)
    return "1" if "1" in (a, b) else "0" if a == b == "0" else "x"


def XOR(a, b):
    a, b = n(a), n(b)
    return "x" if "x" in (a, b) else str(int(a) ^ int(b))


for name, f in (("AND", AND), ("OR", OR), ("XOR", XOR)):
    print(f"{name:4s}  " + "  ".join(V))
    for a in V:
        print(f"  {a}   " + "  ".join(f(a, b) for b in V))
    definite = sum(f(a, b) in "01" for a in V for b in V)
    print(f"  definite results: {definite} of 16\n")
assert [sum(f(a, b) in "01" for a in V for b in V) for f in (AND, OR, XOR)] == [8, 8, 4]

AND   0  1  x  z
  0   0  0  0  0
  1   0  1  x  x
  x   0  x  x  x
  z   0  x  x  x
  definite results: 8 of 16

OR    0  1  x  z
  0   0  1  x  x
  1   1  1  1  1
  x   x  1  x  x
  z   x  1  x  x
  definite results: 8 of 16

XOR   0  1  x  z
  0   0  1  x  x
  1   1  0  x  x
  x   x  x  x  x
  z   x  x  x  x
  definite results: 4 of 16



## Part 2 · Blocking versus non-blocking

A clocked block is a list of assignments. *Blocking* updates each variable immediately; *non-blocking* reads every right-hand side first and updates all at the end.

In [3]:
def clock_edge(stmts, regs, blocking):
    r, pending = dict(regs), {}
    for lhs, rhs in stmts:
        if blocking:
            r[lhs] = r[rhs]
        else:
            pending[lhs] = r[rhs]
    r.update(pending)
    return r


shift = [("q1", "d"), ("q2", "q1"), ("q3", "q2")]
for blocking in (False, True):
    regs = {"d": 1, "q1": 0, "q2": 0, "q3": 0}
    history = []
    for edge in range(1, 5):
        regs = clock_edge(shift, regs, blocking)
        regs["d"] = 0
        history.append((regs["q1"], regs["q2"], regs["q3"]))
    print("blocking    " if blocking else "non-blocking", "q1 q2 q3 after edges 1-4:", history)

print("swap, non-blocking:", clock_edge([("a", "b"), ("b", "a")], {"a": 3, "b": 5}, False))
print("swap, blocking:    ", clock_edge([("a", "b"), ("b", "a")], {"a": 3, "b": 5}, True))

# the race of Problem 08-05.13: two blocks, each with one blocking assignment, in either order
outcomes = set()
for order in itertools.permutations([("x", "y"), ("y", "x")]):
    r = {"x": 0, "y": 1}
    for lhs, rhs in order:
        r[lhs] = r[rhs]
    outcomes.add((r["x"], r["y"]))
print("race outcomes (x, y):", outcomes)
assert outcomes == {(1, 1), (0, 0)}

non-blocking q1 q2 q3 after edges 1-4: [(1, 0, 0), (0, 1, 0), (0, 0, 1), (0, 0, 0)]
blocking     q1 q2 q3 after edges 1-4: [(1, 1, 1), (0, 0, 0), (0, 0, 0), (0, 0, 0)]
swap, non-blocking: {'a': 5, 'b': 3}
swap, blocking:     {'a': 5, 'b': 5}
race outcomes (x, y): {(1, 1), (0, 0)}


## Part 3 · How much does a random test cover? (the chapter's guided exercise)

**🔮 Predict:** with $M$ = 65 536 possible inputs and 65 536 random vectors, how many distinct inputs are hit, and how much does that number vary between seeds?

In [4]:
M = 2 ** 16
q1, q2 = (1 - 1 / M) ** M, (1 - 2 / M) ** M
mean = M * (1 - q1)
sd = math.sqrt(M * q1 * (1 - q1) + M * (M - 1) * (q2 - q1 * q1))
print(f"theory: mean {mean:.1f} distinct inputs hit, standard deviation {sd:.1f}")

counts = [np.unique(rng.integers(0, M, M)).size for _ in range(400)]
print(f"400 simulated runs: mean {np.mean(counts):.1f}, standard deviation {np.std(counts):.1f}")
assert abs(np.mean(counts) - mean) < 5 * sd / math.sqrt(400) * 3

fig, ax = plt.subplots(figsize=(6.4, 3.6))
N = np.arange(0, 6 * M, M // 64)
ax.plot(N / M, 100 * (1 - (1 - 1 / M) ** N), color="#0072B2", label="expected coverage")
seen = np.zeros(M, bool)
draws = rng.integers(0, M, 6 * M)
cov = []
for k in N:
    cov.append(100 * np.unique(draws[:k]).size / M if k else 0)
ax.plot(N / M, cov, "--", color="#D55E00", label="one simulated run")
ax.axvline(1, color="0.6", lw=0.8); ax.axhline(100 * (1 - math.exp(-1)), color="0.6", lw=0.8)
ax.set_xlabel("random vectors / size of the input space, N/M")
ax.set_ylabel("inputs covered (%)")
ax.set_title("Random testing covers 63% at N = M and closes the gap slowly")
ax.legend()
plt.show()

theory: mean 41426.8 distinct inputs hit, standard deviation 79.8


400 simulated runs: mean 41428.3, standard deviation 79.5


The curve reaches 63.2% at $N = M$, 95% only near $N = 3M$, and every input (on average) at $N = M H_M \approx 11.7M$ for this $M$. When exhaustive testing is affordable, it is the cheaper way to full coverage.

## Part 4 · The testbenches, run from here

In [5]:
lab = Path(".")
if HAVE_IVERILOG:
    for tb, designs in (("semantics_tb.v", []), ("cmp8_tb.v", ["cmp8.v"])):
        exe = Path("/tmp") / (tb + ".vvp")
        c = subprocess.run(["iverilog", "-g2012", "-o", str(exe), tb, *designs], capture_output=True, text=True)
        assert c.returncode == 0, c.stderr
        out = subprocess.run(["vvp", "-n", str(exe)], capture_output=True, text=True).stdout
        print(f"--- {tb}\n" + "\n".join(out.strip().splitlines()[-4:]))
        assert "TEST PASSED" in out
else:
    print("Icarus Verilog is not installed here; run `python tools/run_hdl.py` from the repository root instead.")

--- semantics_tb.v
TEST PASSED
semantics_tb.v:131: $finish called at 61000 (1ps)


--- cmp8_tb.v
random run: 41278 of 65536 input pairs hit (24258 missed); expected about 41427
131077 vectors checked
TEST PASSED
cmp8_tb.v:54: $finish called at 131077000 (1ps)


## What to take away

1. X is pessimistic but honest: 0 still dominates AND and 1 still dominates OR.
2. Non-blocking assignments model flip-flops; blocking ones in clocked blocks collapse pipelines and create races.
3. Random testing covers about 63% of a space with as many vectors as the space has inputs, and needs about $M \ln M$ to cover it all: measure coverage, do not assume it.